In [3]:
import os

print("Python is running from:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

Python is running from:
/content

Files in this folder:
['.config', 'sample_data']


In [9]:
import pandas as pd

df = pd.read_csv(
    "twcs.csv",
    on_bad_lines="skip"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset loaded successfully!
Shape: (12275, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [10]:
company_tweets = df[df['inbound'] == False]
brand_counts = company_tweets['author_id'].value_counts()
print(brand_counts.head(20))

author_id
AmazonHelp         522
ChipotleTweets     513
AppleSupport       377
Uber_Support       245
Delta              186
comcastcares       180
AskPlayStation     177
TMobileHelp        176
VerizonSupport     170
SouthwestAir       168
British_Airways    166
hulu_support       166
SpotifyCares       149
Tesco              145
AmericanAir        131
XboxSupport        129
Ask_Spectrum       118
sprintcare         118
AdobeCare          116
AirAsiaSupport     100
Name: count, dtype: int64


In [11]:
brand = "AmericanAir"
brand_ids = df[df['author_id'] == brand]['tweet_id'].tolist()

# Get all tweets that are either from the brand, or replies to/from the brand's threads
# First, get customer tweets that the brand replied to
brand_replies = df[df['author_id'] == brand]
customer_tweet_ids = brand_replies['in_response_to_tweet_id'].dropna().tolist()

customer_tweets = df[df['tweet_id'].isin(customer_tweet_ids)]

print("Brand tweets:", brand_replies.shape)
print("Customer tweets they replied to:", customer_tweets.shape)

Brand tweets: (131, 7)
Customer tweets they replied to: (130, 7)


In [12]:
pairs = brand_replies.merge(
    df,
    left_on='in_response_to_tweet_id',
    right_on='tweet_id',
    suffixes=('_reply', '_customer')
)

pairs = pairs[[
    'tweet_id_customer', 'text_customer',
    'tweet_id_reply', 'text_reply',
    'created_at_customer', 'created_at_reply'
]]

print(pairs.shape)
pairs.head(5)

(130, 6)


,tweet_id_customer,text_customer,tweet_id_reply,text_reply,created_at_customer,created_at_reply
0,997,@AmericanAir Erica on the lax team is amazing ...,996,@115904 We'll be sure to pass along your kind ...,Tue Oct 31 22:24:57 +0000 2017,Tue Oct 31 22:27:30 +0000 2017
1,999,@AmericanAir Could you have someone on your la...,998,@115904 Our apologies for the delay in respond...,Tue Oct 31 21:47:45 +0000 2017,Tue Oct 31 22:12:14 +0000 2017
2,1002,Ben Tennyson and an American Airlines pilot. 🎃...,1001,"@115905 Aww, that's definitely a future pilot ...",Tue Oct 31 22:02:04 +0000 2017,Tue Oct 31 22:24:05 +0000 2017
3,1005,"I’m sorry, what? It’s going to COST me $50 to ...",1003,@115906 This is a great option for customers w...,Tue Oct 31 21:51:37 +0000 2017,Tue Oct 31 22:22:37 +0000 2017
4,1004,"@AmericanAir Right, but I earned those. I also...",1006,@115906 We're sorry for your frustration.,Tue Oct 31 22:24:48 +0000 2017,Tue Oct 31 22:44:51 +0000 2017


In [13]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

sample = pairs.sample(40, random_state=42)
for i, row in sample.iterrows():
    print(f"--- {i} ---")
    print("CUSTOMER:", row['text_customer'])
    print("REPLY:   ", row['text_reply'])
    print()

--- 55 ---
CUSTOMER: Today is the day that @AmericanAir lost my business for the rest of my life as well as everyone in my family.
REPLY:    @116684 Whoa, what's going on that you're feeling this way towards us?

--- 40 ---
CUSTOMER: @AmericanAir I had to end my trip way sooner than expected. :( Bought a new flight because it was way cheaper than changing my return. What do I need to do to cancel my return so someone can have that seat?
REPLY:    @116573 Hi, Matt, we've your info but we'd like to verify which trip you'd like to cancel. Please DM your record locator with the unwanted return.

--- 19 ---
CUSTOMER: Big thx 2 @AmericanAir 4 guacamole and margs of thx while the @116147 lounge upgrades are underway #nomnom https://t.co/PCW4Vdxbyf
REPLY:    @116146 We're happy when you're happy! That sure does look delicious.

--- 31 ---
CUSTOMER: @AmericanAir Yes I see that. Your app says the incoming flight arrives at 10:05 so how are we going to take off at 10:10? https://t.co/vZxyxOa1xo
R

In [14]:
INTENTS = {
    "praise": "Compliments, thanks, positive feedback — no action needed",
    "flight_disruption": "Delays, cancellations, mechanical issues",
    "baggage_issue": "Carry-on/checked bag disputes, fees",
    "staff_complaint": "Rudeness or poor treatment by staff",
    "refund_compensation": "Requests for money back, vouchers, reimbursement",
    "seating_booking": "Seat assignment, upgrades, booking changes",
    "general_other": "Vague venting, app issues, policy questions, off-topic/banter"
}

In [15]:
labeling_batch = pairs.sample(50, random_state=7)
labeling_batch = labeling_batch.reset_index(drop=True)
labeling_batch[['text_customer', 'text_reply']]

,text_customer,text_reply
0,Anyone who needs to book a flight for our wedding: DO IT NOW! Round trip only $85 with @AmericanAir 💍✈️,@116683 Congrats on getting married! We're looking forward to having lots of excited wedding guests. That's a fantastic deal!
1,@AmericanAir Will it be complimentary or charged the 25$ or however much it Is?,@117181 You'll get charged just like a normal checked bag and the charge amount depends on your ticket. Please DM your record locator so we can see.
2,When I email @AmericanAir I get the best customer service. Why the people on the phone aren't enabled to do the same just baffles me.,"@116682 Our service should be consistent all across the board, Stuart. So sorry if you experienced otherwise."
3,LAS - PHL 734 1135am @AmericanAir not 1 smile between the whole crew &amp; your terminal for PHL-DUB flight is awful!,@117155 We have some of the best crew in the business and want you to know that we're glad to have you fly with us today. #smilesfromustoyou!
4,@AmericanAir flight people at the gate or not communicating to the gate area passengers,"@119479 It looks like there's a maintenance delay and we've an estimated departure time of 10:40p. We'll get you in the air soon, Bennett."
5,@AmericanAir @116462 Long-standing problem that AA Mgmt. seems unwilling to address. Screw your EP #RoadWarriors becoz we have no options. Just wait.,@118406 The experience of our most loyal customers is paramount. We're installing a new type of much faster Wi-Fi on much of our fleet.
6,@AmericanAir Really bad service Ron,"@118406 We're sorry if you had a bad experience with our team, Kay. Please let us know if we ca help in some way,"
7,Pleasant experiences with domestic air travel are vanishingly rare. I usually fly @AmericanAir but tried @SouthwestAir —&gt; it was great. Thx,@119246 We want all your travels to be pleasant with us. Please let us know what we can do to help.
8,HAPPY HALLOWEEN!!!!! @AmericanAir https://t.co/kHH0Je6frQ,@118855 What great costumes! Thanks for this fantastic pic.
9,@AmericanAir thanks for awesome flight home #Dreamliner https://t.co/ArQkp3oZh2,@116578 She sure is a gorgeous bird!


In [16]:
labels = [
    ("praise","auto"),("general_other","auto"),("staff_complaint","escalate"),
    ("staff_complaint","escalate"),("flight_disruption","auto"),("general_other","escalate"),
    ("staff_complaint","escalate"),("general_other","auto"),("praise","auto"),
    ("praise","auto"),("baggage_issue","escalate"),("praise","auto"),
    ("praise","auto"),("praise","auto"),("staff_complaint","escalate"),
    ("general_other","auto"),("flight_disruption","escalate"),("general_other","auto"),
    ("general_other","auto"),("staff_complaint","escalate"),("general_other","auto"),
    ("refund_compensation","escalate"),("seating_booking","escalate"),("general_other","auto"),
    ("praise","auto"),("general_other","escalate"),("refund_compensation","escalate"),
    ("general_other","auto"),("baggage_issue","escalate"),("praise","auto"),
    ("praise","auto"),("flight_disruption","auto"),("refund_compensation","escalate"),
    ("praise","auto"),("seating_booking","escalate"),("praise","auto"),
    ("general_other","escalate"),("general_other","auto"),("general_other","escalate"),
    ("staff_complaint","escalate"),("staff_complaint","escalate"),("staff_complaint","escalate"),
    ("praise","auto"),("staff_complaint","escalate"),("general_other","escalate"),
    ("general_other","escalate"),("flight_disruption","auto"),("general_other","escalate"),
    ("refund_compensation","escalate"),("staff_complaint","escalate")
]

labeling_batch["intent"] = [l[0] for l in labels]
labeling_batch["escalation"] = [l[1] for l in labels]

# Save as your golden set file (first batch)
labeling_batch.to_csv("golden_set_batch1.csv", index=False)
print("Saved:", labeling_batch.shape)
labeling_batch[['text_customer', 'intent', 'escalation']].head()

Saved: (50, 8)


,text_customer,intent,escalation
0,Anyone who needs to book a flight for our wedding: DO IT NOW! Round trip only $85 with @AmericanAir 💍✈️,praise,auto
1,@AmericanAir Will it be complimentary or charged the 25$ or however much it Is?,general_other,auto
2,When I email @AmericanAir I get the best customer service. Why the people on the phone aren't enabled to do the same just baffles me.,staff_complaint,escalate
3,LAS - PHL 734 1135am @AmericanAir not 1 smile between the whole crew &amp; your terminal for PHL-DUB flight is awful!,staff_complaint,escalate
4,@AmericanAir flight people at the gate or not communicating to the gate area passengers,flight_disruption,auto


In [17]:
labeling_batch2 = pairs.sample(50, random_state=21)
labeling_batch2 = labeling_batch2.reset_index(drop=True)
labeling_batch2[['text_customer', 'text_reply']]

,text_customer,text_reply
0,Big thx 2 @AmericanAir 4 guacamole and margs of thx while the @116147 lounge upgrades are underway #nomnom https://t.co/PCW4Vdxbyf,@116146 We're happy when you're happy! That sure does look delicious.
1,"@AmericanAir if I have the platinum select aadvantage card and I purchased an int’l ticket through your site but it ended up being on British Air, will I still retain the same benefits of my aadvantage card on this flight (I.e, early boarding)?","@116575 You wouldn't get those perks on BA, Martin."
2,@AmericanAir Shame on you and your company,@116571 We're going to forward this to our flight service leadership team for appropriate review.
3,Anyone who needs to book a flight for our wedding: DO IT NOW! Round trip only $85 with @AmericanAir 💍✈️,@116683 Congrats on getting married! We're looking forward to having lots of excited wedding guests. That's a fantastic deal!
4,@AmericanAir sucks!!!!!! So rude! Left my bag somewhere and they wouldn’t help at all! 🤦🏻‍♀️😠👎🏼 kept telling me they didn’t know why!,@117696 We'd like more info to see if we can help. Please DM your bag file number.
5,Hey @AmericanAir! Rocky here on flight AA2254. We were supposed to depart at 6:07PM but it is now 6:54PM and the crew just informed us that our captain is missing. Oh and not to mention we also had a maintenance issue. Hope to make it to NYC safe and sound. #disappointed,"@116580 We're sorry for the extra travel time today, Rocky. Please DM us your record locator and well take a look."
6,"Thank you, @AmericanAir for playing #ThisIsUs and for having great flight attendants on my flight back home!",@115909 We're glad you got to kick back and enjoy a show while flying! Thanks for your kind words.
7,@AmericanAir Problem is when you check the website compulsively and it’s not updated!,@118743 We understand that's important to update info quickly and we appreciate your feedback on this.
8,Still waiting on @AmericanAir to compensate for all the crap they put @58 and I through on our honeymoon...almost a year ago. 🤔,"@116420 Did you file a report with us? If you did, please send us a DM and include that in so we can take a look."
9,@AmericanAir Will it be complimentary or charged the 25$ or however much it Is?,@117181 You'll get charged just like a normal checked bag and the charge amount depends on your ticket. Please DM your record locator so we can see.


In [18]:
print("Before dedup:", pairs.shape)
pairs = pairs.drop_duplicates(subset=['text_customer', 'text_reply']).reset_index(drop=True)
print("After dedup:", pairs.shape)

Before dedup: (130, 6)
After dedup: (130, 6)


In [19]:
import pandas as pd

# Fresh load
df = pd.read_csv("twcs.csv")
print("Full dataset:", df.shape)

# Filter to American Airlines replies
brand = "AmericanAir"
brand_replies = df[df['author_id'] == brand]
print("AA brand replies:", brand_replies.shape)

# Rebuild pairs
pairs = brand_replies.merge(
    df,
    left_on='in_response_to_tweet_id',
    right_on='tweet_id',
    suffixes=('_reply', '_customer')
)
pairs = pairs[[
    'tweet_id_customer', 'text_customer',
    'tweet_id_reply', 'text_reply',
    'created_at_customer', 'created_at_reply'
]]
print("Pairs before dedup:", pairs.shape)

# Dedup
pairs = pairs.drop_duplicates(subset=['text_customer', 'text_reply']).reset_index(drop=True)
print("Pairs after dedup:", pairs.shape)

Full dataset: (2811774, 7)
AA brand replies: (36764, 7)
Pairs before dedup: (36531, 6)
Pairs after dedup: (36531, 6)


In [8]:
import pandas as pd

# --- Rebuild the verified full dataset + working subsample ---
df = pd.read_csv("twcs.csv")

brand = "AmericanAir"
brand_replies = df[df['author_id'] == brand]

pairs = brand_replies.merge(
    df,
    left_on='in_response_to_tweet_id',
    right_on='tweet_id',
    suffixes=('_reply', '_customer')
)
pairs = pairs[[
    'tweet_id_customer', 'text_customer',
    'tweet_id_reply', 'text_reply',
    'created_at_customer', 'created_at_reply'
]]
pairs = pairs.drop_duplicates(subset=['text_customer', 'text_reply']).reset_index(drop=True)

working_set = pairs.sample(3000, random_state=99).reset_index(drop=True)
working_set.to_csv("aa_working_subsample.csv", index=False)
print("Working subsample:", working_set.shape)

# --- Batch 2: sampled first (random_state=21) ---
labeling_batch2 = working_set.sample(50, random_state=21).reset_index(drop=True)

labels_batch2 = [
    ("general_other","auto"),("flight_disruption","auto"),("seating_booking","escalate"),
    ("baggage_issue","auto"),("flight_disruption","auto"),("praise","auto"),
    ("staff_complaint","escalate"),("staff_complaint","escalate"),("staff_complaint","escalate"),
    ("general_other","auto"),("seating_booking","escalate"),("staff_complaint","escalate"),
    ("flight_disruption","auto"),("flight_disruption","escalate"),("baggage_issue","escalate"),
    ("seating_booking","escalate"),("flight_disruption","auto"),("baggage_issue","escalate"),
    ("general_other","escalate"),("baggage_issue","escalate"),("general_other","auto"),
    ("refund_compensation","escalate"),("general_other","auto"),("staff_complaint","escalate"),
    ("staff_complaint","escalate"),("flight_disruption","auto"),("baggage_issue","escalate"),
    ("staff_complaint","escalate"),("flight_disruption","escalate"),("refund_compensation","escalate"),
    ("general_other","auto"),("staff_complaint","escalate"),("flight_disruption","escalate"),
    ("praise","auto"),("general_other","escalate"),("general_other","auto"),
    ("flight_disruption","auto"),("staff_complaint","escalate"),("general_other","escalate"),
    ("praise","auto"),("general_other","escalate"),("praise","auto"),
    ("general_other","auto"),("staff_complaint","escalate"),("praise","auto"),
    ("general_other","auto"),("praise","auto"),("flight_disruption","escalate"),
    ("praise","auto"),("praise","auto")
]
labeling_batch2["intent"] = [l[0] for l in labels_batch2]
labeling_batch2["escalation"] = [l[1] for l in labels_batch2]
labeling_batch2.to_csv("golden_set_batch2.csv", index=False)
print("Batch 2 saved:", labeling_batch2.shape)

# --- Batch 1: sampled from the pool EXCLUDING batch 2's rows (random_state=7) ---
already_used = labeling_batch2["text_customer"].tolist()
pool = working_set[~working_set["text_customer"].isin(already_used)]
labeling_batch1 = pool.sample(50, random_state=7).reset_index(drop=True)

labels_batch1 = [
    ("general_other","auto"),("flight_disruption","auto"),("staff_complaint","escalate"),
    ("staff_complaint","escalate"),("flight_disruption","auto"),("flight_disruption","auto"),
    ("general_other","escalate"),("praise","auto"),("flight_disruption","auto"),
    ("refund_compensation","escalate"),("general_other","auto"),("general_other","escalate"),
    ("praise","auto"),("staff_complaint","escalate"),("praise","auto"),
    ("seating_booking","escalate"),("flight_disruption","auto"),("flight_disruption","auto"),
    ("general_other","escalate"),("general_other","escalate"),("baggage_issue","escalate"),
    ("praise","auto"),("baggage_issue","auto"),("praise","auto"),
    ("general_other","escalate"),("general_other","escalate"),("praise","auto"),
    ("baggage_issue","auto"),("flight_disruption","auto"),("praise","auto"),
    ("general_other","auto"),("seating_booking","escalate"),("praise","auto"),
    ("seating_booking","escalate"),("praise","auto"),("praise","auto"),
    ("flight_disruption","auto"),("flight_disruption","auto"),("praise","auto"),
    ("praise","auto"),("general_other","auto"),("flight_disruption","auto"),
    ("staff_complaint","escalate"),("flight_disruption","auto"),("staff_complaint","escalate"),
    ("general_other","auto"),("seating_booking","escalate"),("baggage_issue","auto"),
    ("general_other","auto"),("praise","auto")
]
labeling_batch1["intent"] = [l[0] for l in labels_batch1]
labeling_batch1["escalation"] = [l[1] for l in labels_batch1]
labeling_batch1.to_csv("golden_set_batch1.csv", index=False)
print("Batch 1 saved:", labeling_batch1.shape)

# --- Verify ---
combined = pd.concat([labeling_batch1, labeling_batch2], ignore_index=True)
overlap = set(labeling_batch1["text_customer"]) & set(labeling_batch2["text_customer"])
print("\nOverlap check:", len(overlap), "(should be 0)")
print("Combined golden set:", combined.shape)
print("\nIntent distribution:")
print(combined["intent"].value_counts())
print("\nEscalation distribution:")
print(combined["escalation"].value_counts())

Working subsample: (3000, 6)
Batch 2 saved: (50, 8)
Batch 1 saved: (50, 8)

Overlap check: 0 (should be 0)
Combined golden set: (100, 8)

Intent distribution:
intent
general_other          24
flight_disruption      21
praise                 21
staff_complaint        15
baggage_issue           9
seating_booking         7
refund_compensation     3
Name: count, dtype: int64

Escalation distribution:
escalation
auto        56
escalate    44
Name: count, dtype: int64


In [30]:
!pip install openai -q

import openai
import pandas as pd
import json
import getpass

api_key = getpass.getpass("MY_API_KEY")

client = openai.OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

# --- Load golden set ---
batch1 = pd.read_csv("golden_set_batch1.csv")
batch2 = pd.read_csv("golden_set_batch2.csv")
golden = pd.concat([batch1, batch2], ignore_index=True)
print("Golden set:", golden.shape)

INTENTS = {
    "praise": "Compliments, thanks, positive feedback — no action needed",
    "flight_disruption": "Delays, cancellations, mechanical issues",
    "baggage_issue": "Carry-on/checked bag disputes, fees",
    "staff_complaint": "Rudeness or poor treatment by staff",
    "refund_compensation": "Requests for money back, vouchers, reimbursement",
    "seating_booking": "Seat assignment, upgrades, booking changes",
    "general_other": "Vague venting, app issues, policy questions, off-topic/banter"
}

# --- Few-shot examples: 2 per intent, pulled from golden set ---
few_shot_examples = []
for intent in INTENTS:
    examples = golden[golden["intent"] == intent].head(2)
    for _, row in examples.iterrows():
        few_shot_examples.append((row["text_customer"], intent))

few_shot_text = "\n".join([f'Message: "{msg}"\nIntent: {label}' for msg, label in few_shot_examples])

SYSTEM_PROMPT = f"""You are classifying customer support tweets sent to American Airlines into exactly one intent category.

Categories:
{json.dumps(INTENTS, indent=2)}

Here are labeled examples:
{few_shot_text}

Respond with ONLY the intent key (e.g., "flight_disruption"). No explanation, no punctuation."""

MODEL = "google/gemini-2.5-flash"  # OpenRouter's model naming — cheap, fast, good enough for this task

def classify_intent(message):
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=20,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f'Message: "{message}"\nIntent:'}
        ]
    )
    pred = response.choices[0].message.content.strip().lower()
    return pred if pred in INTENTS else "general_other"

# --- Quick test on ONE example first ---
test_message = golden.iloc[0]["text_customer"]
print("Test message:", test_message)
print("Predicted:", classify_intent(test_message))

KeyboardInterrupt: Interrupted by user

In [4]:
# --- Split: exclude few-shot examples from the test set ---
few_shot_texts = [msg for msg, _ in few_shot_examples]
test_set = golden[~golden["text_customer"].isin(few_shot_texts)].reset_index(drop=True)
print("Held-out test set (excludes few-shot examples):", test_set.shape)

# --- Run classifier on held-out test set only ---
test_set["predicted_intent"] = test_set["text_customer"].apply(classify_intent)
test_set["correct"] = test_set["predicted_intent"] == test_set["intent"]

accuracy = test_set["correct"].mean()
print(f"\nLLM Classifier Accuracy (held-out): {accuracy:.2%}")

# --- Baseline 1: Trivial — always predict majority class ---
majority_class = golden["intent"].value_counts().idxmax()
test_set["baseline_majority"] = majority_class
baseline_majority_acc = (test_set["baseline_majority"] == test_set["intent"]).mean()
print(f"Baseline (majority class = '{majority_class}'): {baseline_majority_acc:.2%}")

# --- Baseline 2: Simple keyword matching ---
KEYWORD_RULES = {
    "baggage_issue": ["bag", "luggage", "suitcase", "carry-on", "carry on"],
    "flight_disruption": ["delay", "cancel", "mechanical", "late", "tarmac"],
    "refund_compensation": ["refund", "compensat", "voucher", "money back"],
    "staff_complaint": ["rude", "attendant", "staff", "employee"],
    "seating_booking": ["seat", "upgrade", "book", "reservation"],
    "praise": ["thank", "great", "awesome", "love", "amazing"],
}

def keyword_classify(message):
    msg_lower = message.lower()
    for intent, keywords in KEYWORD_RULES.items():
        if any(kw in msg_lower for kw in keywords):
            return intent
    return "general_other"

test_set["baseline_keyword"] = test_set["text_customer"].apply(keyword_classify)
baseline_keyword_acc = (test_set["baseline_keyword"] == test_set["intent"]).mean()
print(f"Baseline (keyword matching): {baseline_keyword_acc:.2%}")

# --- Save results ---
test_set.to_csv("classifier_results.csv", index=False)
print("\nSaved to classifier_results.csv")

Held-out test set (excludes few-shot examples): (86, 8)

LLM Classifier Accuracy (held-out): 41.86%
Baseline (majority class = 'general_other'): 25.58%
Baseline (keyword matching): 30.23%

Saved to classifier_results.csv


In [5]:
# --- See what's actually going wrong ---
errors = test_set[~test_set["correct"]][["text_customer", "intent", "predicted_intent"]]
print(f"Total errors: {len(errors)} out of {len(test_set)}\n")

# Which true labels get confused with which predicted labels most often?
confusion = errors.groupby(["intent", "predicted_intent"]).size().sort_values(ascending=False)
print("Most common confusions (true → predicted):")
print(confusion.head(15))

print("\n--- Sample misclassified examples ---")
for i, row in errors.head(10).iterrows():
    print(f"TRUE: {row['intent']:20} PREDICTED: {row['predicted_intent']:20}")
    print(f"  MSG: {row['text_customer'][:120]}")
    print()

Total errors: 50 out of 86

Most common confusions (true → predicted):
intent             predicted_intent 
general_other      flight_disruption    5
staff_complaint    flight_disruption    4
flight_disruption  general_other        3
general_other      praise               3
flight_disruption  staff_complaint      3
baggage_issue      general_other        2
                   staff_complaint      2
praise             flight_disruption    2
                   general_other        2
general_other      staff_complaint      2
flight_disruption  praise               2
seating_booking    general_other        2
staff_complaint    baggage_issue        2
seating_booking    flight_disruption    2
staff_complaint    general_other        2
dtype: int64

--- Sample misclassified examples ---
TRUE: general_other        PREDICTED: staff_complaint     
  MSG: Losing one’s wallet is one of the most stressful things. Being told, ain’t shit you can do for 4 hours is worse. Thanks 

TRUE: seating_booking 

In [12]:
from sklearn.metrics import classification_report

print(classification_report(
    errors["intent"],
    errors["predicted_intent"],
    zero_division=0
))

                     precision    recall  f1-score   support

      baggage_issue       0.00      0.00      0.00       6.0
  flight_disruption       0.00      0.00      0.00      10.0
      general_other       0.00      0.00      0.00      11.0
             praise       0.00      0.00      0.00       7.0
refund_compensation       0.00      0.00      0.00       1.0
    seating_booking       0.00      0.00      0.00       5.0
    staff_complaint       0.00      0.00      0.00      10.0

           accuracy                           0.00      50.0
          macro avg       0.00      0.00      0.00      50.0
       weighted avg       0.00      0.00      0.00      50.0



In [10]:
[x for x in globals() if hasattr(globals()[x], "columns")]

['batch1',
 'batch2',
 'golden',
 'examples',
 'test_set',
 'errors',
 'df',
 'brand_replies',
 'pairs',
 'working_set',
 'labeling_batch2',
 'pool',
 'labeling_batch1',
 'combined']

In [13]:
print(errors.columns.tolist())

['text_customer', 'intent', 'predicted_intent']


In [14]:
print(combined.columns.tolist())
print(combined.shape)

['tweet_id_customer', 'text_customer', 'tweet_id_reply', 'text_reply', 'created_at_customer', 'created_at_reply', 'intent', 'escalation']
(100, 8)


In [15]:
for name in ['batch1', 'batch2', 'golden', 'examples', 'test_set',
             'working_set', 'labeling_batch1', 'labeling_batch2', 'pool']:
    d = globals()[name]
    print(name, d.shape, d.columns.tolist())

batch1 (50, 8) ['tweet_id_customer', 'text_customer', 'tweet_id_reply', 'text_reply', 'created_at_customer', 'created_at_reply', 'intent', 'escalation']
batch2 (50, 8) ['tweet_id_customer', 'text_customer', 'tweet_id_reply', 'text_reply', 'created_at_customer', 'created_at_reply', 'intent', 'escalation']
golden (100, 8) ['tweet_id_customer', 'text_customer', 'tweet_id_reply', 'text_reply', 'created_at_customer', 'created_at_reply', 'intent', 'escalation']
examples (2, 8) ['tweet_id_customer', 'text_customer', 'tweet_id_reply', 'text_reply', 'created_at_customer', 'created_at_reply', 'intent', 'escalation']
test_set (86, 12) ['tweet_id_customer', 'text_customer', 'tweet_id_reply', 'text_reply', 'created_at_customer', 'created_at_reply', 'intent', 'escalation', 'predicted_intent', 'correct', 'baseline_majority', 'baseline_keyword']
working_set (3000, 6) ['tweet_id_customer', 'text_customer', 'tweet_id_reply', 'text_reply', 'created_at_customer', 'created_at_reply']
labeling_batch1 (50, 8

In [16]:
from sklearn.metrics import classification_report

print(classification_report(
    test_set["intent"],
    test_set["predicted_intent"],
    zero_division=0
))

                     precision    recall  f1-score   support

      baggage_issue       0.25      0.14      0.18         7
  flight_disruption       0.39      0.47      0.43        19
      general_other       0.50      0.50      0.50        22
             praise       0.57      0.63      0.60        19
refund_compensation       0.00      0.00      0.00         1
    seating_booking       0.00      0.00      0.00         5
    staff_complaint       0.30      0.23      0.26        13

           accuracy                           0.42        86
          macro avg       0.29      0.28      0.28        86
       weighted avg       0.41      0.42      0.41        86



In [17]:
print("LLM accuracy:", test_set["correct"].mean())
print("Majority baseline:", (test_set["intent"] == test_set["baseline_majority"]).mean())
print("Keyword baseline:", (test_set["intent"] == test_set["baseline_keyword"]).mean())

LLM accuracy: 0.4186046511627907
Majority baseline: 0.2558139534883721
Keyword baseline: 0.3023255813953488


In [18]:
print("Golden set distribution:")
print(golden["intent"].value_counts())

print("\nTest set distribution:")
print(test_set["intent"].value_counts())

Golden set distribution:
intent
general_other          24
flight_disruption      21
praise                 21
staff_complaint        15
baggage_issue           9
seating_booking         7
refund_compensation     3
Name: count, dtype: int64

Test set distribution:
intent
general_other          22
flight_disruption      19
praise                 19
staff_complaint        13
baggage_issue           7
seating_booking         5
refund_compensation     1
Name: count, dtype: int64


In [19]:
errors = test_set[~test_set["correct"]][["text_customer", "intent", "predicted_intent"]]

In [20]:
test_set["predicted_intent"]

,predicted_intent
0,flight_disruption
1,flight_disruption
2,general_other
3,staff_complaint
4,staff_complaint
...,...
81,general_other
82,general_other
83,general_other
84,seating_booking


In [21]:
pre_accuracy = test_set["correct"].mean()

print("PRE LLM accuracy:", f"{pre_accuracy:.4f}")
print("PRE LLM accuracy:", f"{pre_accuracy*100:.2f}%")

PRE LLM accuracy: 0.4186
PRE LLM accuracy: 41.86%


In [22]:
for intent in INTENTS:
    print(f"\n=== {intent} ===")
    examples = golden[golden["intent"] == intent].head(2)
    for _, row in examples.iterrows():
        print("-", row["text_customer"])


=== praise ===
- Love the style of this old logo on a vintage @americanair plane. ✈️ .
.
.
.
#aviation #vintage #americanairlines #… https://t.co/jWSfVSlPOc https://t.co/f9xjC73Wge
- @AmericanAir great flight 1620 Denver to Dallas Kelly great flight attendant really professional. Thx

=== flight_disruption ===
- @AmericanAir at gate we were told to go to a16 from a34.  Please advise. Monitors say a34 despite this directive.
- @AmericanAir I hope I don’t miss my flight because it’s impossible to understand the @americanair person on the loudspeaker @ gate 31 @ JFK

=== baggage_issue ===
- It’s ridiculous that @AmericanAir lost my luggage and nobody on the phone has been able to locate it STILL; 3 days later. How. 🙄
- @AmericanAir .. landed in Charlotte at 7:11 PM. It’s now 7:52 without any luggage.. or staff in baggage claim.. or belt movement. Hello? https://t.co/PFxh5m9JIl

=== staff_complaint ===
- @6660 - x4 times on the phone - see an agent at the airport is the reply - @AmericanA

In [23]:
print(golden["intent"].value_counts())

intent
general_other          24
flight_disruption      21
praise                 21
staff_complaint        15
baggage_issue           9
seating_booking         7
refund_compensation     3
Name: count, dtype: int64


In [24]:
golden[golden["intent"] == "refund_compensation"][["text_customer", "text_reply", "intent"]].to_string(index=False)

"                                                                                                                                       text_customer                                                                                                                                           text_reply              intent\n@AmericanAir Thanks but you should provide some comp to all your clients - miles or something- especially those of us who are loyal AdvantageA mbrs!                                                                                   @477663 We're happy to take a peek. Please DM your record locator. refund_compensation\n                                  @122128 has awful baggage service. 30 min after my @AmericanAir flight parks, still no bag. 👎#Lackluster #Slacking @371798 What? You still haven't received your bag. We don't mind looking and getting an update. Send us a quick DM with your record locator, please. refund_compensation\n                                          

In [25]:
print(
    golden[golden["intent"] == "refund_compensation"]
    [["text_customer", "text_reply"]]
    .to_string(index=False)
)

                                                                                                                                       text_customer                                                                                                                                           text_reply
@AmericanAir Thanks but you should provide some comp to all your clients - miles or something- especially those of us who are loyal AdvantageA mbrs!                                                                                   @477663 We're happy to take a peek. Please DM your record locator.
                                  @122128 has awful baggage service. 30 min after my @AmericanAir flight parks, still no bag. 👎#Lackluster #Slacking @371798 What? You still haven't received your bag. We don't mind looking and getting an update. Send us a quick DM with your record locator, please.
                                              I rarely take @AmericanAir but yesterday’s experience convin

In [26]:
for intent in INTENTS:
    print(f"\n{'='*20} {intent} {'='*20}")
    rows = golden[golden["intent"] == intent]
    for i, (_, row) in enumerate(rows.iterrows(), 1):
        print(f"{i}. {row['text_customer']}")


==================== praise ====================
1. Love the style of this old logo on a vintage @americanair plane. ✈️ .
.
.
.
#aviation #vintage #americanairlines #… https://t.co/jWSfVSlPOc https://t.co/f9xjC73Wge
2. @AmericanAir great flight 1620 Denver to Dallas Kelly great flight attendant really professional. Thx
3. @AmericanAir I have flown with your airline the last 3 years and it’s been stellar every single time. #AAwesome #AAvacation
4. Great day for a flight with @AmericanAir https://t.co/kkbeo84jM1
5. Some people are testy! Some people are rude! But they're all well handled! By your professional crew! Thanks, @AmericanAir for getting us home!
6. @AmericanAir thank you for getting me home 6 hours early today!  #Standby
7. @AmericanAir I travel every week for business no isues
8. @AmericanAir Good morning! Always exceptional service with you guys! Could I get a follow back please?
9. I'm trying MCE on my next @AmericanAir flight from RDU to PHL on A319. Can't wait!
10. My pl

In [ ]:
SYSTEM_PROMPT = """You are an intent classifier for customer-support messages sent to American Airlines.

Classify each customer message into EXACTLY ONE of these intents:

praise:
Positive feedback, compliments, thanks, or appreciation.

flight_disruption:
Delays, cancellations, diversions, mechanical problems, deplaning, or being stuck on the runway/tarmac.

baggage_issue:
Lost, missing, delayed, damaged, or mishandled baggage, including baggage fees and carry-on problems.

staff_complaint:
Complaints about rude, unhelpful, disrespectful, or poor behavior by airline employees or customer-service staff.

refund_compensation:
Explicit requests for refunds, reimbursement, compensation, miles, vouchers, or money.
A delay, baggage problem, or complaint is NOT refund_compensation unless compensation/refund is explicitly requested.

seating_booking:
Seat assignment, seat selection, standby status, booking, reservation changes, or booking-related problems.

general_other:
Anything that does not clearly fit the categories above, including general questions, vague complaints, loyalty/mileage questions, Wi-Fi/policy questions, or unclear messages.

DECISION RULES:
- Choose the customer's MAIN issue.
- If multiple issues appear, choose the most central issue.
- Explicit refund/compensation request -> refund_compensation.
- Lost/missing/damaged baggage -> baggage_issue unless explicit compensation is the main request.
- Delay/cancellation/mechanical/diversion/deplaning -> flight_disruption unless explicit compensation is the main request.
- Rude or poor employee behavior -> staff_complaint when that is the main complaint.
- Seat selection, standby, booking, or reservation problems -> seating_booking.
- Pure positive feedback -> praise.
- Otherwise -> general_other.

Respond with ONLY one of these exact intent keys:

praise
flight_disruption
baggage_issue
staff_complaint
refund_compensation
seating_booking
general_other
"""

print("✅ New POST prompt loaded successfully.")

In [31]:
!pip install scikit-learn -q

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load your full pairs data (the historical grounding source)
pairs = pd.read_csv("aa_working_subsample.csv")

# Load golden set (test cases)
b1 = pd.read_csv("golden_set_batch1.csv")
b2 = pd.read_csv("golden_set_batch2.csv")
golden = pd.concat([b1, b2], ignore_index=True)

# --- Build retrieval index over historical AA replies ---
vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')
history_vectors = vectorizer.fit_transform(pairs['text_customer'])

def retrieve_similar_cases(message, k=3):
    query_vec = vectorizer.transform([message])
    sims = cosine_similarity(query_vec, history_vectors)[0]
    top_idx = sims.argsort()[-k:][::-1]
    return pairs.iloc[top_idx][['text_customer', 'text_reply']]

# --- Reply drafter: grounded in retrieved historical replies ---
def draft_reply(message, intent):
    similar = retrieve_similar_cases(message, k=3)
    grounding = "\n".join([f'- Customer said: "{r.text_customer[:100]}" | AA replied: "{r.text_reply}"'
                            for r in similar.itertuples()])

    prompt = f"""You are American Airlines' customer support agent. Draft a reply to this customer message, in AA's typical tone (empathetic, brief, professional, often asks for DM/record locator for real issues).

Customer message: "{message}"
Detected intent: {intent}

Here are similar past exchanges for tone/style reference:
{grounding}

Write ONLY the reply text, nothing else."""

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

# --- Escalation logic: rule-based, grounded in what real AA agents did ---
DM_SIGNALS = ["dm", "direct message", "record locator", "please send", "please share",
              "phone number", "contact you", "reach out", "follow and"]

def decide_escalation(message, intent, drafted_reply):
    # Rule: money/compensation, staff complaints, and baggage issues default to escalate
    # (matches what we found: these categories needed individual case handling in real data)
    high_risk_intents = {"refund_compensation", "staff_complaint", "baggage_issue"}

    if intent in high_risk_intents:
        return "escalate", f"Intent '{intent}' historically required individual case handling by AA agents"

    # Rule: if our own drafted reply asks for DM/record locator, that signals real escalation need
    if any(signal in drafted_reply.lower() for signal in DM_SIGNALS):
        return "escalate", "Reply requires customer to provide case-specific details (DM/record locator)"

    return "auto", f"Intent '{intent}' is typically self-contained (info/apology/praise response)"

# --- Test on a few examples first ---
sample = golden.sample(3, random_state=1)
for _, row in sample.iterrows():
    reply = draft_reply(row['text_customer'], row['intent'])
    decision, reason = decide_escalation(row['text_customer'], row['intent'], reply)
    print(f"CUSTOMER: {row['text_customer'][:100]}")
    print(f"INTENT: {row['intent']}")
    print(f"DRAFTED REPLY: {reply}")
    print(f"DECISION: {decision} — {reason}")
    print(f"GROUND TRUTH ESCALATION: {row['escalation']}")
    print("---")

CUSTOMER: @126403 at @151271 are an absolute nightmare! Thanks @AmericanAir for getting me out of here!
INTENT: general_other
DRAFTED REPLY: Thanks for flying with us! We appreciate you choosing American.
DECISION: auto — Intent 'general_other' is typically self-contained (info/apology/praise response)
GROUND TRUTH ESCALATION: auto
---
CUSTOMER: @AmericanAir Your terrible service never fails to amaze me! Bought this ticket in JAN and service ha
INTENT: general_other
DRAFTED REPLY: We're sorry to hear this, and we'd like to help if we can. Please DM us your record locator and more details about what happened.
DECISION: escalate — Reply requires customer to provide case-specific details (DM/record locator)
GROUND TRUTH ESCALATION: escalate
---
CUSTOMER: @AmericanAir Yep - that doesn’t work. You hold for 30 minutes for them to say they can’t help you fo
INTENT: seating_booking
DRAFTED REPLY: We're sorry to hear you're having trouble with your seating. Please send us a DM with your record 

In [32]:
results = []
for _, row in golden.iterrows():
    reply = draft_reply(row['text_customer'], row['intent'])
    decision, reason = decide_escalation(row['text_customer'], row['intent'], reply)
    results.append({
        "text_customer": row['text_customer'],
        "intent": row['intent'],
        "drafted_reply": reply,
        "predicted_escalation": decision,
        "escalation_reason": reason,
        "true_escalation": row['escalation']
    })

results_df = pd.DataFrame(results)
results_df["escalation_correct"] = results_df["predicted_escalation"] == results_df["true_escalation"]

escalation_accuracy = results_df["escalation_correct"].mean()
print(f"Escalation decision accuracy: {escalation_accuracy:.2%}")
print(f"\nBreakdown:")
print(pd.crosstab(results_df["true_escalation"], results_df["predicted_escalation"]))

results_df.to_csv("full_pipeline_results.csv", index=False)
print("\nSaved to full_pipeline_results.csv")

Escalation decision accuracy: 61.00%

Breakdown:
predicted_escalation  auto  escalate
true_escalation                     
auto                    22        34
escalate                 5        39

Saved to full_pipeline_results.csv


In [33]:
JUDGE_PROMPT_TEMPLATE = """You are evaluating an AI-drafted customer support reply for American Airlines.

Customer message: "{customer_msg}"
AI-drafted reply: "{drafted_reply}"
Real AA agent's actual historical reply (reference, not gold standard): "{reference_reply}"

Rate the AI-drafted reply on a scale of 1-5 for:
- Appropriateness (does it address the actual issue?)
- Tone (professional, empathetic, matches AA's style?)
- Actionability (clear next step if needed?)

Respond in this exact format, nothing else:
SCORE: <1-5>
REASON: <one short sentence>"""

def judge_reply(customer_msg, drafted_reply, reference_reply):
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        customer_msg=customer_msg, drafted_reply=drafted_reply, reference_reply=reference_reply
    )
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=60,
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.choices[0].message.content.strip()
    try:
        score_line = [l for l in text.split("\n") if l.startswith("SCORE")][0]
        score = int(score_line.split(":")[1].strip())
    except:
        score = None
    return score, text

# Load full results, merge in reference replies from golden set
full_results = pd.read_csv("full_pipeline_results.csv")
full_results = full_results.merge(golden[["text_customer", "text_reply"]], on="text_customer", how="left")

# Judge a subsample (30 examples) to keep it fast — judging all 100 isn't necessary
judge_sample = full_results.sample(30, random_state=5).reset_index(drop=True)

judge_scores = []
for _, row in judge_sample.iterrows():
    score, raw = judge_reply(row['text_customer'], row['drafted_reply'], row['text_reply'])
    judge_scores.append(score)

judge_sample["judge_score"] = judge_scores
judge_sample.to_csv("judged_replies.csv", index=False)

print("Average judge score:", judge_sample["judge_score"].mean())
print(judge_sample["judge_score"].value_counts().sort_index())

Average judge score: 4.066666666666666
judge_score
1     1
3     5
4    14
5    10
Name: count, dtype: int64


In [42]:
import pandas as pd
from scipy.stats import pearsonr

# ============================================
# PART 1: LLM-as-Judge scores AI-drafted replies
# ============================================

JUDGE_PROMPT_TEMPLATE = """You are evaluating an AI-drafted customer support reply for American Airlines.

Customer message: "{customer_msg}"
AI-drafted reply: "{drafted_reply}"
Real AA agent's actual historical reply (reference, not gold standard): "{reference_reply}"

Rate the AI-drafted reply on a scale of 1-5 for:
- Appropriateness (does it address the actual issue?)
- Tone (professional, empathetic, matches AA's style?)
- Actionability (clear next step if needed?)

Respond in this exact format, nothing else:
SCORE: <1-5>
REASON: <one short sentence>"""

def judge_reply(customer_msg, drafted_reply, reference_reply):
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        customer_msg=customer_msg, drafted_reply=drafted_reply, reference_reply=reference_reply
    )
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=60,
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.choices[0].message.content.strip()
    try:
        score_line = [l for l in text.split("\n") if l.startswith("SCORE")][0]
        score = int(score_line.split(":")[1].strip())
    except:
        score = None
    return score

full_results = pd.read_csv("full_pipeline_results.csv")
full_results = full_results.merge(golden[["text_customer", "text_reply"]], on="text_customer", how="left")

judge_sample = full_results.sample(30, random_state=5).reset_index(drop=True)
judge_sample["judge_score"] = judge_sample.apply(
    lambda row: judge_reply(row['text_customer'], row['drafted_reply'], row['text_reply']), axis=1
)
judge_sample.to_csv("judged_replies.csv", index=False)

print(f"Average judge score: {judge_sample['judge_score'].mean():.2f}")
print(judge_sample["judge_score"].value_counts().sort_index())

# ============================================
# PART 2: Human-agreement check
# A human rater (project author) blind-scored 12 of these
# same replies WITHOUT seeing the judge's scores, to test
# whether the LLM-judge is a trustworthy proxy for human judgment.
# ============================================

human_check = judge_sample.sample(12, random_state=3).reset_index(drop=True)

# NOTE: these scores were collected via manual blind human review —
# hardcoded here since the rating itself was done outside the notebook.
# See report Section "Judge Reliability" for methodology.
human_scores = [3, 4, 4, 4, 5, 4, 3, 1, 4, 5, 3, 3]

human_check["human_score"] = human_scores
human_check["diff"] = (human_check["human_score"] - human_check["judge_score"]).abs()

exact_match = (human_check["diff"] == 0).mean()
within_1 = (human_check["diff"] <= 1).mean()
corr, pval = pearsonr(human_check["human_score"], human_check["judge_score"])

print("\n=== Human-Judge Agreement ===")
print(f"Exact match rate: {exact_match:.2%}")
print(f"Within 1 point: {within_1:.2%}")
print(f"Pearson correlation: {corr:.3f} (p={pval:.3f})")
print(f"Mean absolute difference: {human_check['diff'].mean():.2f}")

human_check.to_csv("human_agreement_check.csv", index=False)
print("\nSaved: judged_replies.csv, human_agreement_check.csv")

Average judge score: 4.03
judge_score
3.0     8
4.0    12
5.0     9
Name: count, dtype: int64

=== Human-Judge Agreement ===
Exact match rate: 50.00%
Within 1 point: 83.33%
Pearson correlation: 0.702 (p=0.011)
Mean absolute difference: 0.67

Saved: judged_replies.csv, human_agreement_check.csv


In [43]:
# Sanity check: run POST classifier on ONE example only
sample_msg = test_set.iloc[0]["text_customer"]
sample_true = test_set.iloc[0]["intent"]

pred = classify_intent(sample_msg)

print("Message:", sample_msg)
print("True intent:", sample_true)
print("POST predicted intent:", pred)

Message: Really @AmericanAir? Your website (which you helpfully sent me) shows departure at 9:15, even though it's 9:16, and we still have not boarded. The flight was originally 9:00. https://t.co/aZHSWFB9UF
True intent: flight_disruption
POST predicted intent: flight_disruption


In [44]:
# Full POST evaluation on the fixed 86-row test set
post_predictions = []

for idx, row in test_set.iterrows():
    pred = classify_intent(row["text_customer"])
    post_predictions.append(pred)

test_set["predicted_intent_post"] = post_predictions
test_set["correct_post"] = test_set["predicted_intent_post"] == test_set["intent"]

post_accuracy = test_set["correct_post"].mean()
print(f"POST LLM accuracy: {post_accuracy*100:.2f}%")
print(f"PRE  LLM accuracy: 41.86%")
print(f"Majority baseline: 25.58%")
print(f"Keyword baseline:  30.23%")

POST LLM accuracy: 41.86%
PRE  LLM accuracy: 41.86%
Majority baseline: 25.58%
Keyword baseline:  30.23%


In [45]:
# Are POST and PRE predictions actually different?
same_as_pre = (test_set["predicted_intent_post"] == test_set["predicted_intent"])
print(f"Rows where POST prediction == PRE prediction: {same_as_pre.sum()} / {len(test_set)}")

# Show any rows that differ
diffs = test_set[~same_as_pre][["text_customer", "intent", "predicted_intent", "predicted_intent_post"]]
print(f"\nNumber of differing predictions: {len(diffs)}")
diffs.head(20)


Rows where POST prediction == PRE prediction: 79 / 86

Number of differing predictions: 7


,text_customer,intent,predicted_intent,predicted_intent_post
23,Well this is going to be fun. 35 mins to go an...,flight_disruption,staff_complaint,general_other
44,@AmericanAir They wouldn't hold the plane for ...,staff_complaint,staff_complaint,flight_disruption
46,@AmericanAir Tried to change it on the sly wit...,general_other,flight_disruption,seating_booking
54,@AmericanAir Approaching an hour since we land...,baggage_issue,flight_disruption,baggage_issue
64,@AmericanAir how do I get my KTN added to my b...,flight_disruption,seating_booking,general_other
67,@AmericanAir Way to make your passengers comfo...,staff_complaint,general_other,praise
82,@215340 &amp; @AmericanAir be ashamed of the p...,praise,general_other,staff_complaint


In [46]:
from sklearn.metrics import classification_report

print("=== POST classification report ===")
print(classification_report(
    test_set["intent"],
    test_set["predicted_intent_post"],
    zero_division=0
))

=== POST classification report ===
                     precision    recall  f1-score   support

      baggage_issue       0.40      0.29      0.33         7
  flight_disruption       0.41      0.47      0.44        19
      general_other       0.50      0.50      0.50        22
             praise       0.55      0.63      0.59        19
refund_compensation       0.00      0.00      0.00         1
    seating_booking       0.00      0.00      0.00         5
    staff_complaint       0.22      0.15      0.18        13

           accuracy                           0.42        86
          macro avg       0.30      0.29      0.29        86
       weighted avg       0.40      0.42      0.41        86



In [47]:
# Save POST results alongside PRE (don't overwrite classifier_results.csv)
test_set.to_csv("post_classifier_results.csv", index=False)
print("Saved: post_classifier_results.csv")

# POST error breakdown (same style as PRE section 9)
post_errors = test_set[~test_set["correct_post"]][["text_customer", "intent", "predicted_intent_post"]]
print(f"\n{len(post_errors)} errors out of {len(test_set)}")

confusion_counts = test_set[~test_set["correct_post"]].groupby(["intent", "predicted_intent_post"]).size().sort_values(ascending=False)
print("\nMost common POST confusions:")
print(confusion_counts.head(15))

Saved: post_classifier_results.csv

50 errors out of 86

Most common POST confusions:
intent             predicted_intent_post
flight_disruption  general_other            5
staff_complaint    flight_disruption        5
general_other      flight_disruption        4
                   praise                   3
flight_disruption  staff_complaint          2
baggage_issue      general_other            2
general_other      seating_booking          2
flight_disruption  praise                   2
general_other      staff_complaint          2
baggage_issue      staff_complaint          2
seating_booking    general_other            2
staff_complaint    baggage_issue            2
seating_booking    flight_disruption        2
praise             flight_disruption        2
staff_complaint    praise                   2
dtype: int64


In [48]:
# Failure mode 1: staff_complaint mistaken for flight_disruption (worsened POST vs PRE)
fm1 = test_set[(test_set["intent"]=="staff_complaint") & (test_set["predicted_intent_post"]=="flight_disruption")]
print("=== staff_complaint -> flight_disruption ===")
for t in fm1["text_customer"]:
    print("-", t[:200])

# Failure mode 2: general_other used as catch-all, things pulled OUT of general_other into flight_disruption
fm2 = test_set[(test_set["intent"]=="general_other") & (test_set["predicted_intent_post"]=="flight_disruption")]
print("\n=== general_other -> flight_disruption ===")
for t in fm2["text_customer"]:
    print("-", t[:200])

=== staff_complaint -> flight_disruption ===
- Really @AmericanAir and @Delta you both almost made me miss my flight.  Running in #clt to find this flight 😤 who is responsible and how? https://t.co/oYsmIfe3Bs
- @AmericanAir They wouldn't hold the plane for 3 minutes when I ran over there from K10 to K8. Door was already closed. Had to spend the night in CHI after waiting on  a long, slow Customer Service lin
- Get your act together @AmericanAir. Been sitting on the ground @ LHR for 2 hrs.
- This is going to be the second plane I have to get off because something is wrong w/@AmericanAir planes.! #NEVERAGAIN #AA5 #AmericanAirlines
- @AmericanAir By the way, my original flight out this morning was also delayed causing me to miss my connection in Phoenix.  Second delayed flight.  There is a pattern.

=== general_other -> flight_disruption ===
- @AmericanAir We’ve just been told we have to sit here and wait until the next flight at 8:30am! No hotels, no other flights...this is truly the wor

In [49]:
# Failure mode candidates 3-5: pull examples for the other big confusion buckets
fm3 = test_set[(test_set["intent"]=="general_other") & (test_set["predicted_intent_post"]=="praise")]
print("=== general_other -> praise ===")
for t in fm3["text_customer"]:
    print("-", t[:200])

fm4 = test_set[(test_set["intent"]=="baggage_issue") & (test_set["predicted_intent_post"].isin(["general_other","staff_complaint"]))]
print("\n=== baggage_issue -> general_other/staff_complaint ===")
for t in fm4["text_customer"]:
    print("-", t[:200])

fm5 = test_set[(test_set["intent"]=="seating_booking") & (test_set["predicted_intent_post"].isin(["general_other","flight_disruption"]))]
print("\n=== seating_booking -> general_other/flight_disruption ===")
for t in fm5["text_customer"]:
    print("-", t[:200])

=== general_other -> praise ===
- @Americanair Chelsea at @46428 gate C21 was an amazing gate agent.  Friendly, helpful, no upgrade today but super helpful attitude
- @126403 at @151271 are an absolute nightmare! Thanks @AmericanAir for getting me out of here!
- Flying @AmericanAir from now on! https://t.co/V6sFe611gL

=== baggage_issue -> general_other/staff_complaint ===
- @AmericanAir currently in a rebooking line.. you pretended to link my Member # to my AA Platinum Select Card #CmonSon #AmericanAirlines 🤔
- @AmericanAir Thanks! It's A4576 PHL to CMS at 5:35
- @AmericanAir Not a good time for your canned remarks. Lines are long. People are frustrated and your PHL employees are always rude.
- Viajar con @AmericanAir es una pesadilla, será que es un requisito que sus sobrecargos sean unos groseros y flojos ?

=== seating_booking -> general_other/flight_disruption ===
- @AmericanAir Yep - that doesn’t work. You hold for 30 minutes for them to say they can’t help you for flights booked

In [50]:
# Full POST evaluation on the fixed 86-row test set
post_predictions = []

for idx, row in test_set.iterrows():
    pred = classify_intent(row["text_customer"])
    post_predictions.append(pred)

test_set["predicted_intent_post"] = post_predictions
test_set["correct_post"] = test_set["predicted_intent_post"] == test_set["intent"]

post_accuracy = test_set["correct_post"].mean()
print(f"POST LLM accuracy: {post_accuracy*100:.2f}%")
print(f"PRE  LLM accuracy: 41.86%")
print(f"Majority baseline: 25.58%")
print(f"Keyword baseline:  30.23%")

POST LLM accuracy: 44.19%
PRE  LLM accuracy: 41.86%
Majority baseline: 25.58%
Keyword baseline:  30.23%


In [51]:
from sklearn.metrics import classification_report

print(classification_report(
    test_set["intent"],
    test_set["predicted_intent_post"],
    zero_division=0,
    digits=2
))

# POST confusions, most common first
errors_post = test_set[~test_set["correct_post"]]
print("\nPOST errors:", len(errors_post), "of", len(test_set))
print(errors_post.groupby(["intent", "predicted_intent_post"]).size()
      .sort_values(ascending=False).head(15))

# Rows PRE got right and POST got wrong - the regressions
regressions = test_set[test_set["correct"] & ~test_set["correct_post"]]
print("\nRegressions (PRE right, POST wrong):", len(regressions))
for _, r in regressions.head(5).iterrows():
    print("-", r["text_customer"][:140])
    print("   gold:", r["intent"], "| PRE:", r["predicted_intent"], "| POST:", r["predicted_intent_post"])

                     precision    recall  f1-score   support

      baggage_issue       0.40      0.29      0.33         7
  flight_disruption       0.45      0.53      0.49        19
      general_other       0.55      0.50      0.52        22
             praise       0.55      0.63      0.59        19
refund_compensation       0.00      0.00      0.00         1
    seating_booking       0.17      0.20      0.18         5
    staff_complaint       0.22      0.15      0.18        13

           accuracy                           0.44        86
          macro avg       0.33      0.33      0.33        86
       weighted avg       0.44      0.44      0.44        86


POST errors: 48 of 86
intent             predicted_intent_post
general_other      flight_disruption        4
staff_complaint    flight_disruption        4
flight_disruption  general_other            3
general_other      praise                   3
flight_disruption  staff_complaint          2
baggage_issue      general_other

In [52]:
test_set.to_csv("classifier_results.csv", index=False)
test_set.to_csv("test_set.csv", index=False)
golden.to_csv("golden_set.csv", index=False)   # 100-row version; step 4 replaces this

from google.colab import files
files.download("classifier_results.csv")
files.download("test_set.csv")
files.download("golden_set.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [53]:
import pandas as pd

topup = pd.read_csv("golden_set_topup_suggested.csv")
cols = ["tweet_id_customer", "text_customer", "tweet_id_reply", "text_reply",
        "created_at_customer", "created_at_reply", "intent", "escalation"]
topup = topup[cols]

golden_180 = pd.concat([golden[cols], topup], ignore_index=True)
golden_180 = golden_180.drop_duplicates(subset="tweet_id_customer")

print(golden_180.shape)
print(golden_180["intent"].value_counts())
print(golden_180["escalation"].value_counts())

golden_180.to_csv("golden_set.csv", index=False)
from google.colab import files
files.download("golden_set.csv")

(180, 8)
intent
general_other          52
praise                 41
flight_disruption      36
staff_complaint        19
baggage_issue          16
seating_booking        10
refund_compensation     6
Name: count, dtype: int64
escalation
auto        103
escalate     77
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
import pandas as pd

full = pd.read_csv("golden_set_180_full.csv")  # upload this file to Colab first

preds = []
for _, row in full.iterrows():
    pred = classify_intent(row["text_customer"])
    preds.append(pred)

full["model_predicted_intent"] = preds
full["model_agrees_with_label"] = full["model_predicted_intent"] == full["intent"]

print(f"Model agrees with your labels on {full['model_agrees_with_label'].mean()*100:.1f}% of all 180 golden rows")
print(full.groupby(["intent","model_predicted_intent"]).size().sort_values(ascending=False).head(15))

full.to_csv("golden_set_180_full.csv", index=False)
from google.colab import files
files.download("golden_set_180_full.csv")

Model agrees with your labels on 82.5% of all 180 golden rows
intent               model_predicted_intent
general_other        general_other             23
praise               praise                    18
flight_disruption    flight_disruption         12
baggage_issue        baggage_issue              6
staff_complaint      staff_complaint            3
praise               general_other              2
general_other        staff_complaint            2
seating_booking      seating_booking            2
refund_compensation  refund_compensation        2
baggage_issue        flight_disruption          1
flight_disruption    general_other              1
general_other        seating_booking            1
                     praise                     1
flight_disruption    staff_complaint            1
general_other        flight_disruption          1
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [58]:
bad = full[full["intent"].isna() | (full["intent"].str.strip() == "")]
print(bad[["tweet_id_customer","text_customer","intent","model_predicted_intent"]])

Empty DataFrame
Columns: [tweet_id_customer, text_customer, intent, model_predicted_intent]
Index: []


In [61]:
full["model_agrees_with_label"] = full["model_predicted_intent"] == full["intent"]
print(f"{full['model_agrees_with_label'].mean()*100:.1f}%")
full.to_csv("golden_set_180_full.csv", index=False)

82.5%


In [62]:
# Full POST evaluation on the fixed 86-row test set
post_predictions = []

for idx, row in test_set.iterrows():
    pred = classify_intent(row["text_customer"])
    post_predictions.append(pred)

test_set["predicted_intent_post"] = post_predictions
test_set["correct_post"] = test_set["predicted_intent_post"] == test_set["intent"]

post_accuracy = test_set["correct_post"].mean()
print(f"POST LLM accuracy: {post_accuracy*100:.2f}%")
print(f"PRE  LLM accuracy: 41.86%")
print(f"Majority baseline: 25.58%")
print(f"Keyword baseline:  30.23%")

POST LLM accuracy: 43.02%
PRE  LLM accuracy: 41.86%
Majority baseline: 25.58%
Keyword baseline:  30.23%
